# 🌿 Notebook 1 — The Strangler Fig Pattern

> **Goal:** replace a legacy system *without ever doing a big-bang rewrite*.

### The metaphor
A **strangler fig** is a tree that sprouts on the branch of a host tree, drops roots
to the ground, and slowly grows *around* the host. Years later the host has rotted
away and only the fig remains — standing in the exact shape of the tree it replaced.

Martin Fowler borrowed this image in 2004 for software migration:

> *"Gradually create a new system around the edges of the old, letting it grow
> slowly over several years until the old system is strangled."*

### What you'll learn in this notebook
1. Why "just rewrite it" (the big-bang) almost always fails.
2. The three moving parts of every strangler migration: **legacy**, **new**, **facade**.
3. A tiny, runnable facade that routes by path.
4. Real-world stories (Amazon, Shopify, GitHub, Netflix).


## 🛠️ Setup

```bash
cd 05-microservices/strangler
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

> This lab uses **only the Python standard library** — no servers, no databases. Every cell is a small simulation you can read end-to-end.


## 1. ❌ Bad practice: the big-bang rewrite

The tempting plan every team considers at least once:

> *"The legacy code is a mess. Let's freeze features for 6 months, rewrite everything
> from scratch on the new stack, then flip a switch."*

Why it almost never works:

| Problem | What actually happens |
|---|---|
| Moving target | The business keeps shipping features to the old system while you rewrite. You're always behind. |
| Hidden business rules | Legacy code encodes years of edge cases ("customers created before 2014 get free shipping on Tuesdays"). You only discover them *after* cutover. |
| One giant cutover | Day 1 of the new system = maximum risk, maximum data, zero feedback history. |
| No rollback | Once the old DB is off, you can't go back without losing hours/days of data. |
| Morale | 12 months of "no new features" is brutal. People quit. |

Netscape, Friendster, Borland, and Windows Longhorn are the famous public examples
of rewrites that sank the company or the product.

👉 The strangler pattern exists because the big-bang is *so* seductive and *so* risky.

## 2. ✅ Good practice: strangle it slowly

Instead of one cutover, you do **dozens of tiny cutovers**, each behind a switch.

```
  ┌──────────┐        ┌───────────┐        ┌──────────────┐
  │  Client  │ ─────▶ │  Facade   │ ─────▶ │   Legacy     │
  └──────────┘        │ (router)  │        └──────────────┘
                      │           │        ┌──────────────┐
                      │           │ ─────▶ │ New service  │
                      └───────────┘        └──────────────┘
```

Three pieces:

1. **Legacy** — the existing system. *Do not touch it* except to keep the lights on.
2. **New service** — built on the new stack, implementing *one slice* of behaviour.
3. **Facade / router** — sits in front of both. It's the only thing clients talk to.

The facade is the single place where you decide "this path goes to new, that one
still goes to legacy". Over months you flip more and more routes. Eventually the
legacy system handles 0% of traffic and you turn it off.


## 3. 🧪 Runnable example: facade by path

Imagine a small bank. The legacy app handles *everything*. We'll start migrating
the `/users` endpoints to a new service while `/orders` and `/reports` stay on legacy.

In [1]:
# --- Legacy system: assume this is 15 years of PHP we can barely touch ---
def legacy_app(path, payload=None):
    return {"source": "legacy-v1", "path": path, "payload": payload}

# --- New service: shiny, well-tested, only implements /users for now ---
def new_user_service(path, payload=None):
    return {"source": "new-user-svc", "path": path, "payload": payload}

# --- Facade: the gatekeeper. The ONLY place routing decisions live. ---
MIGRATED_PREFIXES = ("/users",)   # start tiny — just one slice

def facade(path, payload=None):
    if path.startswith(MIGRATED_PREFIXES):
        return new_user_service(path, payload)
    return legacy_app(path, payload)

# Try it:
for p in ["/users", "/users/42", "/orders/99", "/reports/daily"]:
    print(f"{p:20s} -> {facade(p)}")


/users               -> {'source': 'new-user-svc', 'path': '/users', 'payload': None}
/users/42            -> {'source': 'new-user-svc', 'path': '/users/42', 'payload': None}
/orders/99           -> {'source': 'legacy-v1', 'path': '/orders/99', 'payload': None}
/reports/daily       -> {'source': 'legacy-v1', 'path': '/reports/daily', 'payload': None}


### What just happened?
- Clients call **only** the facade. They have no idea the backend was split.
- The facade made a *boring* string decision: prefix match.
- To migrate another slice tomorrow, we append one line to `MIGRATED_PREFIXES`.
- To **roll back**, we remove that line. No code in legacy or new-svc changes.

This is the whole pattern. Everything else in this lab is variations on this theme.

## 4. 🗺️ How to pick the first slice

Not every feature is a good first migration target. Choose one that is:

| Criterion | Why it matters |
|---|---|
| **Well-bounded** | Few dependencies on other parts of the legacy system. |
| **Low risk** | If you break it for 5 minutes, nobody loses money. Read-only endpoints are great first picks. |
| **High pain** | Slow, buggy, or expensive to run — so the business supports the work. |
| **Clear contract** | The input/output is stable and easy to re-implement. |

Classic first slices in real migrations:
- A **read-only reporting endpoint** (no writes = safe).
- **Static content / search** that's already mostly cached.
- A **new feature** that doesn't exist in legacy at all — build it in the new service from day one.


## 5. 🌍 Real-world stories

- **Amazon (2001-2006)** — broke the "Obidos" monolith into services one call at a time. Jeff Bezos' famous 2002 "services or you're fired" memo was essentially a strangler mandate.
- **eBay** — moved from a C++ monolith to Java services over ~5 years, route by route.
- **Shopify** — is *still* extracting modules from their Rails monolith ("Modular Monolith → Services") using an internal router.
- **GitHub** — the migration from a Rails monolith toward services (e.g. the Git backend becoming `Spokes`, notifications becoming their own service) uses this pattern.
- **Netflix** — famously strangled their DVD-era datacenter monolith into 500+ AWS services between 2008 and 2016.
- **Stack Overflow** — the opposite direction: strangled *microservices* back into their monolith when the extra complexity wasn't worth it. The pattern works both ways.

The common thread: **years, not months**, and **no downtime**.


## 6. 🧭 The four phases of a strangler migration

1. **Identify** — pick one slice of the legacy system. Document its inputs, outputs, and weird edge cases.
2. **Intercept** — put the facade in front. At this stage it's a no-op: 100% traffic still goes to legacy. This alone is a scary deploy and worth doing first, on its own.
3. **Replace** — build the new service for that one slice. Route a *small* % of traffic to it (see Notebook 2).
4. **Retire** — once 100% of traffic is on the new service and the legacy code hasn't run for weeks, delete it. **Don't skip this step** — unused code rots and lies to you.

> Engineers love steps 1–3 and forget step 4. That's how you end up with 3 systems instead of 1.


## 7. ⚠️ Common pitfalls

- **Migrating data too early.** Move *behaviour* first; share the legacy DB through a thin adapter. Migrate the data in a separate, later project.
- **Facade becomes a second monolith.** Keep the router dumb. Business logic lives in services, not in the router.
- **Forgetting the retirement phase.** If legacy is still running "just in case" two years later, you have two systems to maintain forever.
- **Big-bang disguised as strangler.** "We'll migrate 80 endpoints in one sprint" is not a strangler — it's a big-bang wearing a costume.


### ➡️ Next
Notebook 2 shows the **gradual traffic shift** — the canary percentage roll-out,
dark launches (sending traffic to *both* systems and comparing), and rollback.
Notebook 3 is an end-to-end **worked example**: migrating a banking service with
dual-writes, metrics, and retirement.